## 1. Answer the questions

### 1.1 Выведите аналитическое решение задачи регрессии. Используйте векторную форму уравнения.

- Задача регрессии - задача, в которой нужно предсказать значение переменной (y), на основе одной или нескольких переменных (X)
- Дано: Xi - матрица признаков и соответствующее значение yi

- Векторная форма уравнения: y = Xw + e
    - y - Вектор значений
    - X - Матрица признаков
    - w - Вектор коэффициентов 
    - e - Вектор ошибок

- Цель:
  - Найти вектор коэффициентов w, который минимизирует сумму квадратов ошибок.
  - Ошибки определяются как разность между значениями y и предсказанными значениями Xw

- Минимизация ошибки: Q(X, w) = (Xw - y)² -> min
- Метод Наименьших Квадратов (МНК)

- Для того, чтобы найти минимальное w, нужно раскрыть уравнение  
    (Xw - y)ᵀ(Xw - y) =  
    (Xᵀwᵀ - yᵀ)(Xw - y) =    
    XᵀwᵀXw - Xᵀwᵀy - yᵀXw + yᵀy =      
    yᵀy - 2yᵀXw + wᵀXᵀXw  
- Затем нужно взять производную и приравнять значение к 0, чтобы найти минимальное  
    -2Xᵀy + 2XᵀXw = 0  
    2XᵀXw = 2Xᵀy  
    w = Xᵀy / XᵀX
- Итого:
    w = (XᵀX)⁻¹ * Xᵀy    
- Это псевдообратная матрица 
    
    
  

### 1.2 Что меняется в решении при добавлении регуляризаций L1 и L2 к функции потерь?

- Регуляризация — это техника для предотвращения переобучения модели. Переобучение происходит, когда модель слишком хорошо подстраивается под обучающие данные, включая шум и выбросы, что приводит к плохой обобщающей способности на новых данных.
- Кратко: Регуляризации - это штраф за большие веса.

- L1 Регуляризация (Lasso)
    - Добавляет к функции потерь сумму абсолютных значений коэффициентов модели, умноженную на некоторый коэффициент регуляризации (λ)
    - Loss = Loss(original) + λ∑∣wᵢ∣
    -  Q(X, w) = (Xw - y)² + λ∑∣wᵢ∣
    -  где, wᵢ - коэф модели, λ - параметр регуляризации.

- Свойства L1 Регуляризации:
    - Сжатие коэффициентов: Некоторые коэф станут равны нулю. Происходит отбор наиболее важных признаков
    - Непрерывность: Создает ромбовидный контур уровня, что приводит к тому, что решение может находиться на краю контура, где некоторые коэф будут равны нулю

- L2 Регуляризация (Ridge)
    - Добавляет к функции потерь сумму квадратов значений коэффициентов модели, умноженнную на коэффициент регуляризации (λ)
    - Loss = Loss(original) + λ∑wᵢ²
    - Q(X, w) = (Xw - y)² + λ∑wᵢ²

- Свойства L2 Регуляризации:
    - Сглаживание коэффициентов: L2 регуляризация не приводит к обнулению коэф, а скорее уменьшает их значение. Это в силу круглой формы контуров уровня
    - Устойчивоть: L2 регуляризация помогает избежать переобучения, особенно когда  много признаков, которые могут быть коррелированы. Она "размазывает" влияние всех признаков, что делает модель более устойчивой к шуму.
    - ∑∣wᵢ∣ = ||wᵢ||

### 1.3 Объясните, почему регуляризация L1 часто используется для выбора признаков. Почему после подгонки модели многие веса равны 0?

- Это следует из свойств регуляризации, потому что просиходит отбор наиболее важных признаков, другие зануляются.

- Когда мы минимизируем функцию потерь, L1 регуляризация создает "ромбовидный" контур уровня. Это означает, что оптимальное решение может находиться на границе этого контура, где некоторые коэффициенты равны нулю.

### 1.4 Объясните, как можно использовать те же модели (линейную регрессию, гребневую и т. д.), но сделать возможным подгонку нелинейных зависимостей.

- Полиномиальные признаки (использовать к примеру не только x, а еще x², x³)
- Преобразование признаков (к примеру есть признак x, можно добавить log(x) или sin(x) в качестве новых признаков)
- Взаимодействие признаков (есть признаки x1, x2, можно добавить признак x1 * x2)

## 2. Introduction

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures

## 3. Intro data analysis part 2

In [2]:
df = pd.read_json('data/train.json')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  object 
 3   created          49352 non-null  object 
 4   description      49352 non-null  object 
 5   display_address  49352 non-null  object 
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  object 
 11  photos           49352 non-null  object 
 12  price            49352 non-null  int64  
 13  street_address   49352 non-null  object 
 14  interest_level   49352 non-null  object 
dtypes: float64(3), int64(3), object(9)
memory usage: 6.0+ MB


In [3]:
df['features'] = df['features'].apply(lambda x: [i.replace(' ', '').\
                                                 replace('[', '').\
                                                 replace(']', '').\
                                                 replace('"', '').\
                                                 replace("'", '') for i in x])

In [4]:
df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, LaundryinBuilding, Dishwas...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, LaundryinBuilding, Laundry...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, FitnessCenter, LaundryinBu...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low


In [5]:
result = []

for index, row in df.iterrows():
    values = [value.strip() for value in row['features']]
    result.extend(values)

In [6]:
result[:10]

['DiningRoom',
 'Pre-War',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'DogsAllowed',
 'CatsAllowed',
 'Doorman',
 'Elevator',
 'LaundryinBuilding']

In [7]:
unique_result = set(result)
len(unique_result)

1545

In [8]:
feature_counts = Counter(result)
top20features = feature_counts.most_common(20)
top20features

[('Elevator', 25915),
 ('CatsAllowed', 23540),
 ('HardwoodFloors', 23527),
 ('DogsAllowed', 22035),
 ('Doorman', 20898),
 ('Dishwasher', 20426),
 ('NoFee', 18062),
 ('LaundryinBuilding', 16344),
 ('FitnessCenter', 13252),
 ('Pre-War', 9148),
 ('LaundryinUnit', 8738),
 ('RoofDeck', 6542),
 ('OutdoorSpace', 5268),
 ('DiningRoom', 5136),
 ('HighSpeedInternet', 4299),
 ('Balcony', 2992),
 ('SwimmingPool', 2730),
 ('LaundryInBuilding', 2593),
 ('NewConstruction', 2559),
 ('Terrace', 2283)]

In [9]:
res_df = pd.DataFrame()

In [10]:
for feature in top20features:
    res_df[feature[0]] = df['features'].apply(lambda x: 1 if feature[0] in x else 0)

In [11]:
res_df.iloc[0]

Elevator             0
CatsAllowed          1
HardwoodFloors       1
DogsAllowed          1
Doorman              0
Dishwasher           1
NoFee                0
LaundryinBuilding    1
FitnessCenter        0
Pre-War              1
LaundryinUnit        0
RoofDeck             0
OutdoorSpace         0
DiningRoom           1
HighSpeedInternet    0
Balcony              0
SwimmingPool         0
LaundryInBuilding    0
NewConstruction      0
Terrace              0
Name: 4, dtype: int64

In [12]:
res_df.head()

,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,0,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,0,1,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0


In [13]:
res_df[['bathrooms', 'bedrooms', 'interest_level']] = df[['bathrooms', 'bedrooms', 'interest_level']]

In [14]:
res_df.head()

,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms,interest_level
4,0,1,1,1,0,1,0,1,0,1,...,1,0,0,0,0,0,0,1.0,1,medium
6,1,0,1,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,1.0,2,low
9,1,0,1,0,1,1,0,1,0,0,...,0,0,0,0,0,0,0,1.0,2,medium
10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1.5,3,medium
15,1,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,0,0,1.0,0,low


In [15]:
res_df['interest_level'] = res_df['interest_level'].replace({'low': 1, 'medium': 2, 'high': 3})
res_df.head()

,Elevator,CatsAllowed,HardwoodFloors,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,...,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace,bathrooms,bedrooms,interest_level
4,0,1,1,1,0,1,0,1,0,1,...,1,0,0,0,0,0,0,1.0,1,2
6,1,0,1,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,1.0,2,1
9,1,0,1,0,1,1,0,1,0,0,...,0,0,0,0,0,0,0,1.0,2,2
10,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1.5,3,2
15,1,0,0,0,1,0,0,1,1,0,...,0,0,0,0,0,0,0,1.0,0,1


In [16]:
feature_list = res_df.columns.tolist()
feature_list

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace',
 'bathrooms',
 'bedrooms',
 'interest_level']

## 4. Models implementation — Linear regression

In [17]:
class LinearRegressionSGD:
    def __init__(self, learning_rate=0.1, n_iter=100):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape # Получаю количество образцов и количество признаков
        self.weights = np.zeros(n_features) # Инициализирую значения весов нулями
        self.bias = 0

        for _ in range(self.n_iter):
            for i in range(n_samples):
                linear_output = np.dot(X.iloc[i], self.weights) + self.bias # Предсказание 
                y_pred = linear_output

                dw = (1 / n_samples) * (y_pred - y.iloc[i]) * X.iloc[i] # Градиент по весам
                db = (1 / n_samples) * (y_pred - y.iloc[i]) # Градиент по смещению

                self.weights -= self.learning_rate * dw # Обновление весов 
                self.bias -= self.learning_rate * db # Обновние смещения

    def predict(self, X):
        result = np.dot(X, self.weights) + self.bias
        return result

In [18]:
X, y = res_df, df['price']

In [19]:
class LinearRegressionGD:
    def __init__(self, learning_rate=0.1, n_iter=100):
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape 
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.n_iter):
            y_pred = np.dot(X, self.weights) + self.bias

            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        result = np.dot(X, self.weights) + self.bias
        return result    

In [20]:
class LinearRegressionAnalytic:
    def __init__(self):
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X] # Добавляем столбец единиц к X, чтобы учесть смещение
        best_param = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
        self.weights = best_param[1:] # Сохраняем веса
        self.bias = best_param[0] # Сохраняем смещение

    def predict(self, X):
        result = np.dot(X, self.weights) + self.bias
        return result

Коэффициент детерминации R2(R-квадрат) 
— это статистическая мера, которая показывает, насколько хорошо данные соответствуют модели регрессии. Он варьируется от 0 до 1, где 0 указывает на то, что модель не объясняет вариацию зависимой переменной, а 1 указывает на то, что модель полностью объясняет вариацию.  
- R2 = 1 − (SS(tot) / SS(res))
- где:
    - SS(res) — сумма квадратов остатков (разница между предсказанными и фактическими значениями),
    - SS(tot) — общая сумма квадратов (разница между фактическими значениями и их средним).

In [21]:
def r_squared(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    return r2

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [23]:
modelSGD = LinearRegressionSGD()
modelSGD.fit(X_train, y_train)

In [24]:
modelGD = LinearRegressionGD()
modelGD.fit(X_train, y_train)

In [25]:
modelAnalytic = LinearRegressionAnalytic()
modelAnalytic.fit(X_train, y_train)

In [26]:
modelLinear = LinearRegression()
modelLinear.fit(X_train, y_train)

LinearRegression()

In [27]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'test'])

In [28]:
predict_modelSGD_train = modelSGD.predict(X_train)
predict_modelSGD_test = modelSGD.predict(X_test)

In [29]:
mae_train = mean_absolute_error(y_train, predict_modelSGD_train)
mae_test = mean_absolute_error(y_test, predict_modelSGD_test)
result_MAE.loc[len(result_MAE)] = ['modelSGD', mae_train, mae_test]

In [30]:
predict_modelGD_train = modelGD.predict(X_train)
predict_modelGD_test = modelGD.predict(X_test)

In [31]:
mae_train = mean_absolute_error(y_train, predict_modelGD_train)
mae_test = mean_absolute_error(y_test, predict_modelGD_test)
result_MAE.loc[len(result_MAE)] = ['modelGD', mae_train, mae_test]

In [32]:
predict_modelAnalytic_train = modelAnalytic.predict(X_train)
predict_modelAnalytic_test = modelAnalytic.predict(X_test)

In [33]:
mae_train = mean_absolute_error(y_train, predict_modelAnalytic_train)
mae_test = mean_absolute_error(y_test, predict_modelAnalytic_test)
result_MAE.loc[len(result_MAE)] = ['modelAnalytic', mae_train, mae_test]

In [34]:
predict_modelLinear_train = modelLinear.predict(X_train)
predict_modelLinear_test = modelLinear.predict(X_test)

In [35]:
mae_train = mean_absolute_error(y_train, predict_modelLinear_train)
mae_test = mean_absolute_error(y_test, predict_modelLinear_test)
result_MAE.loc[len(result_MAE)] = ['modelLinear', mae_train, mae_test]

In [36]:
result_MAE

,model,train,test
0,modelSGD,1170.481106,982.454338
1,modelGD,1172.157007,984.170900
2,modelAnalytic,1227.602615,1042.656600
3,modelLinear,1227.602615,1042.656600


In [37]:
result_RMSE = pd.DataFrame(columns=['model', 'train', 'test'])

In [38]:
rmse_train = mean_squared_error(y_train, predict_modelSGD_train)
rmse_test = mean_squared_error(y_test, predict_modelSGD_test)
result_RMSE.loc[len(result_RMSE)] = ['modelSGD', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [39]:
rmse_train = mean_squared_error(y_train, predict_modelGD_train)
rmse_test = mean_squared_error(y_test, predict_modelGD_test)
result_RMSE.loc[len(result_RMSE)] = ['modelGD', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [40]:
rmse_train = mean_squared_error(y_train, predict_modelAnalytic_train)
rmse_test = mean_squared_error(y_test, predict_modelAnalytic_test)
result_RMSE.loc[len(result_RMSE)] = ['modelAnalytic', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [41]:
rmse_train = mean_squared_error(y_train, predict_modelLinear_train)
rmse_test = mean_squared_error(y_test, predict_modelLinear_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLinear', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [42]:
result_RMSE

,model,train,test
0,modelSGD,24565.663785,2186.437729
1,modelGD,24565.647317,2186.888804
2,modelAnalytic,24564.274224,2216.946576
3,modelLinear,24564.274224,2216.946576


In [43]:
result_r2 = pd.DataFrame(columns=['model', 'train', 'test'])

In [44]:
r2_train = r_squared(y_train, predict_modelSGD_train)
r2_test = r_squared(y_test, predict_modelSGD_test)
result_r2.loc[len(result_r2)] = ['modelSGD', r2_train, r2_test]

In [45]:
r2_train = r_squared(y_train, predict_modelGD_train)
r2_test = r_squared(y_test, predict_modelGD_test)
result_r2.loc[len(result_r2)] = ['modelGD', r2_train, r2_test]

In [46]:
r2_train = r_squared(y_train, predict_modelAnalytic_train)
r2_test = r_squared(y_test, predict_modelAnalytic_test)
result_r2.loc[len(result_r2)] = ['modelAnalytic', r2_train, r2_test]

In [47]:
r2_train = r_squared(y_train, predict_modelLinear_train)
r2_test = r_squared(y_test, predict_modelLinear_test)
result_r2.loc[len(result_r2)] = ['modelLinear', r2_train, r2_test]

In [48]:
result_r2

,model,train,test
0,modelSGD,0.005532,0.352032
1,modelGD,0.005533,0.351765
2,modelAnalytic,0.005644,0.333823
3,modelLinear,0.005644,0.333823


## 5. Regularized models implementation — Ridge, Lasso, ElasticNet

In [49]:
class CustomRidge:
    def __init__(self, alpha=1.0):
        self.alpha = alpha # Параметр регуляризации
        self.coef_ = None # Коэф модели
        self.intercept_ = None # Свободный член

    def fit(self, X, y):
        X_b = np.c_[np.ones((X.shape[0], 1)), X] # Добавляем столбец единиц к X, чтобы учесть смещение
        I = np.eye(X_b.shape[1]) # Создаем единичную матрицу
        I[0, 0] = 0  # Не регуляризуем свободный член
        # По формуле для регрессии Риджа, вычисляем коэффициенты модели
        self.coef_ = np.linalg.inv(X_b.T.dot(X_b) + self.alpha * I).dot(X_b.T).dot(y)
        self.intercept_ = self.coef_[0] # Сохраняем значение свободного члена
        self.coef_ = self.coef_[1:] # Сохраняем коэф без смещения

    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [50]:
class CustomLasso:
    def __init__(self, alpha=1.0, lr=0.1, n_iter=100):
        self.alpha = alpha
        self.lr = lr # Learning rate, скорость обучения
        self.n_iter = n_iter # Колчиество итераций
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X, y):
        m, n = X.shape
        self.coef_ = np.zeros(n)
        self.intercept_ = 0
        
        for _ in range(self.n_iter):
            y_pred = np.dot(X, self.coef_) + self.intercept_
            errors = y_pred - y
            
            # Считаем градиенты
            d_coef = np.dot(X.T, errors) / m + self.alpha * np.sign(self.coef_)
            # np.dot(X.T * errors) / m - градиент потерь
            # self.alpha * np.sign(self.coef_) - штраф
            d_intercept = np.mean(errors)

            self.coef_ -= self.lr * d_coef
            self.intercept_ -= self.lr * d_intercept
            
    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [51]:
class CustomElasticNet:
    def __init__(self, alpha=1.0, l1_ratio=0.5, lr=0.01, n_iter=1000):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.lr = lr
        self.n_iter = n_iter
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X, y):
        m, n = X.shape
        self.coef_ = np.zeros(n)
        self.intercept_ = 0

        for _ in range(self.n_iter):
            y_pred = np.dot(X, self.coef_) + self.intercept_
            errors = y_pred - y
            # Градиенты
            l1_term = self.alpha * self.l1_ratio * np.sign(self.coef_)
            l2_term = self.alpha * (1 - self.l1_ratio) * self.coef_
            d_coef = np.dot(X.T, errors) / m + l1_term + l2_term
            d_intercept = np.mean(errors)

            self.coef_ -= self.lr * d_coef
            self.intercept_ -= self.lr * d_intercept

    def predict(self, X):
        return np.dot(X, self.coef_) + self.intercept_

In [52]:
modelRidge = CustomRidge()
modelRidge.fit(X_train, y_train)

In [53]:
predict_modelRidge_train = modelRidge.predict(X_train)
predict_modelRidge_test = modelRidge.predict(X_test)

In [54]:
mae_train = mean_absolute_error(y_train, predict_modelRidge_train)
mae_test = mean_absolute_error(y_test, predict_modelRidge_test)
result_MAE.loc[len(result_MAE)] = ['modelRidge', mae_train, mae_test]

In [55]:
modelLasso = CustomLasso()
modelLasso.fit(X_train, y_train)

In [56]:
predict_modelLasso_train = modelLasso.predict(X_train)
predict_modelLasso_test = modelLasso.predict(X_test)

In [57]:
mae_train = mean_absolute_error(y_train, predict_modelLasso_train)
mae_test = mean_absolute_error(y_test, predict_modelLasso_test)
result_MAE.loc[len(result_MAE)] = ['modelLasso', mae_train, mae_test]

In [58]:
modelElasticNet = CustomElasticNet()
modelElasticNet.fit(X_train, y_train)

In [59]:
predict_ElasticNet_train = modelElasticNet.predict(X_train)
predict_ElasticNet_test = modelElasticNet.predict(X_test)

In [60]:
mae_train = mean_absolute_error(y_train, predict_ElasticNet_train)
mae_test = mean_absolute_error(y_test, predict_ElasticNet_test)
result_MAE.loc[len(result_MAE)] = ['modelElasticNet', mae_train, mae_test]

In [61]:
mae_train = mean_squared_error(y_train, predict_modelRidge_train)
mae_test = mean_squared_error(y_test, predict_modelLasso_test)
result_RMSE.loc[len(result_RMSE)] = ['modelRidge', np.sqrt(mae_train), np.sqrt(mae_test)]

In [62]:
mae_train = mean_squared_error(y_train, predict_modelLasso_train)
mae_test = mean_squared_error(y_test, predict_modelRidge_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLasso', np.sqrt(mae_train), np.sqrt(mae_test)]

In [63]:
mae_train = mean_squared_error(y_train, predict_ElasticNet_train)
mae_test = mean_squared_error(y_test, predict_ElasticNet_test)
result_RMSE.loc[len(result_RMSE)] = ['modelElasticNet', np.sqrt(mae_train), np.sqrt(mae_test)]

In [64]:
result_MAE

,model,train,test
0,modelSGD,1170.481106,982.454338
1,modelGD,1172.157007,984.170900
2,modelAnalytic,1227.602615,1042.656600
3,modelLinear,1227.602615,1042.656600
4,modelRidge,1227.530842,1042.582289
5,modelLasso,1170.303823,982.387609
6,modelElasticNet,1112.349847,921.339524


## 6. Feature normalization

- Нормализация признаков — это процесс приведения всех признаков к одному масштабу. Это важно, потому что многие алгоритмы машинного обучения чувствительны к масштабу данных.
Нормализация обязательна:
    - Методы, основанные на расстояниях
    - Методы, использующие градиентный спуск
    - Методы с регуляризацией
Нормализация не обязательна:
    - Деревья решений и ансамбли на их основе:
    - Методы, основанные на вероятностях:

In [65]:
def custom_minmax_scaler(X):
    X_min = np.min(X, axis=0)
    X_max = np.max(X, axis=0)
    X_scaled = (X - X_min) / (X_max - X_min)
    return X_scaled

In [66]:
X_scaled_custom = custom_minmax_scaler(X_train)
print("Custom MinMaxScaler:\n", X_scaled_custom)

Custom MinMaxScaler:
         Elevator  CatsAllowed  HardwoodFloors  DogsAllowed  Doorman  \
4387         1.0          1.0             1.0          1.0      1.0   
34981        1.0          1.0             1.0          1.0      1.0   
63290        1.0          1.0             1.0          1.0      0.0   
121920       1.0          0.0             1.0          0.0      1.0   
40159        0.0          1.0             0.0          1.0      0.0   
...          ...          ...             ...          ...      ...   
101852       1.0          0.0             1.0          0.0      0.0   
65476        0.0          0.0             1.0          0.0      0.0   
103979       0.0          0.0             0.0          0.0      0.0   
58876        1.0          0.0             0.0          0.0      1.0   
10163        1.0          1.0             0.0          1.0      1.0   

        Dishwasher  NoFee  LaundryinBuilding  FitnessCenter  Pre-War  ...  \
4387           1.0    1.0                1.0    

# X(scaled) = x - x(min) / x(max) - x(min)

In [67]:
scaler = MinMaxScaler()
X_scaled_sklearn = scaler.fit_transform(X_train)
print("Sklearn MinMaxScaler:\n", X_scaled_sklearn)

Sklearn MinMaxScaler:
 [[1.    1.    1.    ... 0.2   0.375 0.   ]
 [1.    1.    1.    ... 0.2   0.25  0.5  ]
 [1.    1.    1.    ... 0.1   0.    0.   ]
 ...
 [0.    0.    0.    ... 0.1   0.25  0.   ]
 [1.    0.    0.    ... 0.1   0.375 0.5  ]
 [1.    1.    0.    ... 0.2   0.25  0.   ]]


In [68]:
np.allclose(X_scaled_custom, X_scaled_sklearn) # Сравнение эквивалентности

True

# z = (x - u) / q

In [69]:
def custom_standard_scaler(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    X_scaled = (X - mean) / std
    return X_scaled

In [70]:
X_scaled_custom = custom_standard_scaler(X_train)
print("Custom StandardScaler:\n", X_scaled_custom)

Custom StandardScaler:
         Elevator  CatsAllowed  HardwoodFloors  DogsAllowed   Doorman  \
4387    0.950636     1.047929        1.048728     1.113521  1.168991   
34981   0.950636     1.047929        1.048728     1.113521  1.168991   
63290   0.950636     1.047929        1.048728     1.113521 -0.855438   
121920  0.950636    -0.954263        1.048728    -0.898052  1.168991   
40159  -1.051928     1.047929       -0.953536     1.113521 -0.855438   
...          ...          ...             ...          ...       ...   
101852  0.950636    -0.954263        1.048728    -0.898052 -0.855438   
65476  -1.051928    -0.954263        1.048728    -0.898052 -0.855438   
103979 -1.051928    -0.954263       -0.953536    -0.898052 -0.855438   
58876   0.950636    -0.954263       -0.953536    -0.898052  1.168991   
10163   0.950636     1.047929       -0.953536     1.113521  1.168991   

        Dishwasher     NoFee  LaundryinBuilding  FitnessCenter   Pre-War  ...  \
4387      1.191853  1.318379  

In [71]:
scaler = StandardScaler()
X_scaled_sklearn = scaler.fit_transform(X_train)
print("Sklearn StandardScaler:\n", X_scaled_sklearn)

Sklearn StandardScaler:
 [[ 0.95063557  1.04792935  1.04872771 ...  1.5736351   1.31017423
  -0.61381243]
 [ 0.95063557  1.04792935  1.04872771 ...  1.5736351   0.41252584
   0.98167961]
 [ 0.95063557  1.04792935  1.04872771 ... -0.42484301 -1.38277096
  -0.61381243]
 ...
 [-1.05192782 -0.9542628  -0.95353636 ... -0.42484301  0.41252584
  -0.61381243]
 [ 0.95063557 -0.9542628  -0.95353636 ... -0.42484301  1.31017423
   0.98167961]
 [ 0.95063557  1.04792935 -0.95353636 ...  1.5736351   0.41252584
  -0.61381243]]


In [72]:
np.allclose(X_scaled_custom, X_scaled_sklearn)

True

In [73]:
X_scaled = custom_minmax_scaler(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

In [74]:
modelRidge = CustomRidge()
modelRidge.fit(X_train, y_train)

In [75]:
predict_modelRidge_train = modelRidge.predict(X_train)
predict_modelRidge_test = modelRidge.predict(X_test)

In [76]:
mae_train = mean_absolute_error(y_train, predict_modelRidge_train)
mae_test = mean_absolute_error(y_test, predict_modelRidge_test)
result_MAE.loc[len(result_MAE)] = ['modelRidge MinMaxScaled', mae_train, mae_test]

In [77]:
mae_train = mean_squared_error(y_train, predict_modelRidge_train)
mae_test = mean_squared_error(y_test, predict_modelLasso_test)
result_RMSE.loc[len(result_RMSE)] = ['modelRidge MinMaxScaled', np.sqrt(mae_train), np.sqrt(mae_test)]

In [78]:
modelLasso = CustomLasso()
modelLasso.fit(X_train, y_train)

In [79]:
predict_modelLasso_train = modelLasso.predict(X_train)
predict_modelLasso_test = modelLasso.predict(X_test)

In [80]:
mae_train = mean_absolute_error(y_train, predict_modelLasso_train)
mae_test = mean_absolute_error(y_test, predict_modelLasso_test)
result_MAE.loc[len(result_MAE)] = ['modelLasso MinMaxScaled', mae_train, mae_test]

In [81]:
mae_train = mean_squared_error(y_train, predict_modelLasso_train)
mae_test = mean_squared_error(y_test, predict_modelRidge_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLasso MinMaxScaled', np.sqrt(mae_train), np.sqrt(mae_test)]

In [82]:
modelElasticNet = CustomElasticNet()
modelElasticNet.fit(X_train, y_train)

In [83]:
predict_ElasticNet_train = modelElasticNet.predict(X_train)
predict_ElasticNet_test = modelElasticNet.predict(X_test)

In [84]:
mae_train = mean_absolute_error(y_train, predict_ElasticNet_train)
mae_test = mean_absolute_error(y_test, predict_ElasticNet_test)
result_MAE.loc[len(result_MAE)] = ['modelElasticNet MinMaxScaled', mae_train, mae_test]

In [85]:
mae_train = mean_squared_error(y_train, predict_ElasticNet_train)
mae_test = mean_squared_error(y_test, predict_ElasticNet_test)
result_RMSE.loc[len(result_RMSE)] = ['modelElasticNet MinMaxScaled', np.sqrt(mae_train), np.sqrt(mae_test)]

In [86]:
modelLinear = LinearRegression()
modelLinear.fit(X_train, y_train)

LinearRegression()

In [87]:
predict_modelLinear_train = modelLinear.predict(X_train)
predict_modelLinear_test = modelLinear.predict(X_test)

In [88]:
mae_train = mean_absolute_error(y_train, predict_ElasticNet_train)
mae_test = mean_absolute_error(y_test, predict_ElasticNet_test)
result_MAE.loc[len(result_MAE)] = ['modelLinear MinMaxScaled', mae_train, mae_test]

In [89]:
rmse_train = mean_squared_error(y_train, predict_modelLinear_train)
rmse_test = mean_squared_error(y_test, predict_modelLinear_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLinear MinMaxScaled', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [90]:
X_stander = custom_standard_scaler(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

In [91]:
modelRidge = CustomRidge()
modelRidge.fit(X_train, y_train)

In [92]:
predict_modelRidge_train = modelRidge.predict(X_train)
predict_modelRidge_test = modelRidge.predict(X_test)

In [93]:
mae_train = mean_absolute_error(y_train, predict_modelRidge_train)
mae_test = mean_absolute_error(y_test, predict_modelRidge_test)
result_MAE.loc[len(result_MAE)] = ['modelRidge StandartScaled', mae_train, mae_test]

In [94]:
mae_train = mean_squared_error(y_train, predict_modelRidge_train)
mae_test = mean_squared_error(y_test, predict_modelLasso_test)
result_RMSE.loc[len(result_RMSE)] = ['modelRidge StandartScaled', np.sqrt(mae_train), np.sqrt(mae_test)]

In [95]:
modelLasso = CustomLasso()
modelLasso.fit(X_train, y_train)

In [96]:
predict_modelLasso_train = modelLasso.predict(X_train)
predict_modelLasso_test = modelLasso.predict(X_test)

In [97]:
mae_train = mean_absolute_error(y_train, predict_modelLasso_train)
mae_test = mean_absolute_error(y_test, predict_modelLasso_test)
result_MAE.loc[len(result_MAE)] = ['modelLasso StandartScaled', mae_train, mae_test]

In [98]:
mae_train = mean_absolute_error(y_train, predict_modelLasso_train)
mae_test = mean_absolute_error(y_test, predict_modelLasso_test)
result_MAE.loc[len(result_MAE)] = ['modelLasso StandartScaled', mae_train, mae_test]

In [99]:
modelElasticNet = CustomElasticNet()
modelElasticNet.fit(X_train, y_train)

In [100]:
predict_ElasticNet_train = modelElasticNet.predict(X_train)
predict_ElasticNet_test = modelElasticNet.predict(X_test)

In [101]:
mae_train = mean_absolute_error(y_train, predict_ElasticNet_train)
mae_test = mean_absolute_error(y_test, predict_ElasticNet_test)
result_MAE.loc[len(result_MAE)] = ['modelElasticNet StandartScaled', mae_train, mae_test]

In [102]:
mae_train = mean_squared_error(y_train, predict_ElasticNet_train)
mae_test = mean_squared_error(y_test, predict_ElasticNet_test)
result_RMSE.loc[len(result_RMSE)] = ['modelElasticNet StandartScaled', np.sqrt(mae_train), np.sqrt(mae_test)]

In [103]:
modelLinear = LinearRegression()
modelLinear.fit(X_train, y_train)

LinearRegression()

In [104]:
predict_modelLinear_train = modelLinear.predict(X_train)
predict_modelLinear_test = modelLinear.predict(X_test)

In [105]:
mae_train = mean_absolute_error(y_train, predict_ElasticNet_train)
mae_test = mean_absolute_error(y_test, predict_ElasticNet_test)
result_MAE.loc[len(result_MAE)] = ['modelLinear StandartScaled', mae_train, mae_test]

In [106]:
rmse_train = mean_squared_error(y_train, predict_modelLinear_train)
rmse_test = mean_squared_error(y_test, predict_modelLinear_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLinear StandartScaled', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [107]:
result_MAE

,model,train,test
0,modelSGD,1170.481106,982.454338
1,modelGD,1172.157007,984.170900
2,modelAnalytic,1227.602615,1042.656600
3,modelLinear,1227.602615,1042.656600
4,modelRidge,1227.530842,1042.582289
5,modelLasso,1170.303823,982.387609
6,modelElasticNet,1112.349847,921.339524
7,modelRidge MinMaxScaled,976.809459,1351.253025
8,modelLasso MinMaxScaled,1185.229963,1560.947889
9,modelElasticNet MinMaxScaled,1315.492779,1688.715857


In [108]:
poly = PolynomialFeatures(degree=10)
df['interest_level'] = df['interest_level'].replace({'low': 1, 'medium': 2, 'high': 3})
df.head()

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,2
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, LaundryinBuilding, Dishwas...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,1
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, LaundryinBuilding, Laundry...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,2
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,2
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, FitnessCenter, LaundryinBu...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,1


In [109]:
X_poly = poly.fit_transform(df[['bedrooms', 'bathrooms', 'interest_level']])

In [110]:
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.2, random_state=42)

In [111]:
modelLinear = LinearRegression()
modelLinear.fit(X_train, y_train)

LinearRegression()

In [112]:
predict_modelLinear_train = modelLinear.predict(X_train)
predict_modelLinear_test = modelLinear.predict(X_test)

In [113]:
mae_train = mean_absolute_error(y_train, predict_modelLinear_train)
mae_test = mean_absolute_error(y_test, predict_modelLinear_test)
result_MAE.loc[len(result_MAE)] = ['modelLinear Polynomial', mae_train, mae_test]

In [114]:
rmse_train = mean_squared_error(y_train, predict_modelLinear_train)
rmse_test = mean_squared_error(y_test, predict_modelLinear_test)
result_RMSE.loc[len(result_RMSE)] = ['modelLinear Polynomial', np.sqrt(rmse_train), np.sqrt(rmse_test)]

In [115]:
modelRidge = CustomRidge()
modelRidge.fit(X_train, y_train)

In [116]:
predict_modelRidge_train = modelRidge.predict(X_train)
predict_modelRidge_test = modelRidge.predict(X_test)

In [117]:
mae_train = mean_absolute_error(y_train, predict_modelRidge_train)
mae_test = mean_absolute_error(y_test, predict_modelRidge_test)
result_MAE.loc[len(result_MAE)] = ['modelRidge Polynomial', mae_train, mae_test]

In [118]:
mae_train = mean_squared_error(y_train, predict_modelRidge_train)
mae_test = mean_squared_error(y_test, predict_modelLasso_test)
result_RMSE.loc[len(result_RMSE)] = ['modelRidge Polynomial', np.sqrt(mae_train), np.sqrt(mae_test)]

In [119]:
modelElasticNet = CustomElasticNet()
modelElasticNet.fit(X_train, y_train)

In [120]:
predict_ElasticNet_train = modelElasticNet.predict(X_train)
predict_ElasticNet_test = modelElasticNet.predict(X_test)

In [121]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [122]:
train_mean = y_train.median()
test_mean = y_test.median()

In [123]:
train_median = y_train.median()
test_median = y_test.median()

In [124]:
mean_df = pd.DataFrame({'mean_value': [train_mean] * len(y_train)})
result_MAE.loc[len(result_MAE)] = ['mean', train_mean, test_mean]

In [125]:
result_MAE.loc[len(result_MAE)] = ['median', train_median, test_median]

In [126]:
result_RMSE.loc[len(result_MAE)] = ['median', train_median, test_median]

In [127]:
result_RMSE.loc[len(result_MAE)] = ['mean', train_mean, test_mean]

In [128]:
result_MAE

,model,train,test
0,modelSGD,1170.481106,982.454338
1,modelGD,1172.157007,984.170900
2,modelAnalytic,1227.602615,1042.656600
3,modelLinear,1227.602615,1042.656600
4,modelRidge,1227.530842,1042.582289
5,modelLasso,1170.303823,982.387609
6,modelElasticNet,1112.349847,921.339524
7,modelRidge MinMaxScaled,976.809459,1351.253025
8,modelLasso MinMaxScaled,1185.229963,1560.947889
9,modelElasticNet MinMaxScaled,1315.492779,1688.715857


In [129]:
result_RMSE

,model,train,test
0,modelSGD,24565.663785,2186.437729
1,modelGD,24565.647317,2186.888804
2,modelAnalytic,24564.274224,2216.946576
3,modelLinear,24564.274224,2216.946576
4,modelRidge,24564.274226,2186.208890
5,modelLasso,24565.690396,2216.906329
6,modelElasticNet,24582.027218,2267.172323
7,modelRidge MinMaxScaled,9721.292895,45228.029468
8,modelLasso MinMaxScaled,9801.011761,45173.076485
9,modelElasticNet MinMaxScaled,9841.321122,45208.037094


## Лучшая модель modelRidge MinMaxScaled и modelLasso MinMaxScaled